# Task 3: Primary or Metastatic Inference

This notebook contains the inference reproduction code for Task 3 - primary or metastatic. 

## Prerequisites

1. Please install the environment according to the instructions in README.md
2. Launch the corresponding bonecot environment in Jupyter
3. Ensure you have the required model checkpoints and data prepared

This code will perform 5-fold cross-validation inference for primary or metastatic task.


## Step 0: Check if data, labels and checkpoints are ready

Before running inference, we need to verify that all required files are in place:
- Dataset files and labels for each fold
- Pre-trained model checkpoints (BoneFM backbone and fold-specific classifiers)
- Output directories for saving results 


In [1]:
import os
import pandas as pd
import pathlib

def check_data_labels_checkpoints():
    """
    Check if all required data, labels and checkpoints are ready for Task 3 primary or metastatic inference
    """
    print("Checking data, labels and checkpoints...")
    
    # Get absolute paths
    current_dir = pathlib.Path().absolute()
    finetune_dir = current_dir / "finetune"
    
    print(f"Current directory: {current_dir}")
    print(f"Finetune directory: {finetune_dir}")
    
    # Check if finetune directory exists
    assert finetune_dir.exists(), f"Finetune directory does not exist: {finetune_dir}"
    
    # Check datasets for each fold (fold_0 to fold_4)
    datasets_dir = finetune_dir / "data" / "datasets"
    assert datasets_dir.exists(), f"Datasets directory does not exist: {datasets_dir}"
    
    for fold_num in range(5):
        fold_dir = datasets_dir / f"fold_{fold_num}"
        assert fold_dir.exists(), f"Fold directory does not exist: {fold_dir}"
        
        # Check if each fold has exactly 125 subdirectories
        subdirs = [d for d in fold_dir.iterdir() if d.is_dir()]
        assert len(subdirs) == 125, f"Fold {fold_num} should have 125 subdirectories, but has {len(subdirs)}"
        print(f"✓ Fold {fold_num} dataset: {len(subdirs)} subdirectories found")
    
    # Check labels for task3_primary_or_metastatic
    labels_dir = finetune_dir / "data" / "labels" / "task3_primary_or_metastatic"
    assert labels_dir.exists(), f"Labels directory does not exist: {labels_dir}"
    
    for fold_num in range(5):
        csv_file = labels_dir / f"fold_{fold_num}.csv"
        assert csv_file.exists(), f"Label file does not exist: {csv_file}"
        
        # Read CSV and check if img_path files exist
        df = pd.read_csv(csv_file)
        assert 'img_path' in df.columns, f"img_path column not found in {csv_file}"
        
        print(f"Checking image paths for fold {fold_num} ({len(df)} entries)...")
        missing_files = []
        
        for idx, img_path in enumerate(df['img_path']):
            # Convert relative path to absolute path
            if not os.path.isabs(img_path):
                img_full_path = current_dir / img_path
            else:
                img_full_path = pathlib.Path(img_path)
            
            if not img_full_path.exists():
                missing_files.append(str(img_full_path))
        
        assert len(missing_files) == 0, f"Missing {len(missing_files)} image files in fold {fold_num}. First few missing: {missing_files[:5]}"
        print(f"✓ Fold {fold_num} labels: All {len(df)} image files exist")
    
    # Check model checkpoints
    checkpoints_dir = finetune_dir / "checkpoints"
    assert checkpoints_dir.exists(), f"Checkpoints directory does not exist: {checkpoints_dir}"
    
    # Check BoneFM backbone checkpoint
    backbone_checkpoint = checkpoints_dir / "BoneFM.pth"
    assert backbone_checkpoint.exists(), f"BoneFM backbone checkpoint does not exist: {backbone_checkpoint}"
    print(f"✓ BoneFM backbone checkpoint found")
    
    # Check task3_primary_or_metastatic fold-specific checkpoints
    task3_primary_or_metastatic_dir = checkpoints_dir / "task3_primary_or_metastatic"
    assert task3_primary_or_metastatic_dir.exists(), f"task3_primary_or_metastatic checkpoint directory does not exist: {task3_primary_or_metastatic_dir}"
    
    for fold_num in range(5):
        fold_checkpoint = task3_primary_or_metastatic_dir / f"fold_{fold_num}.pth"
        assert fold_checkpoint.exists(), f"Fold {fold_num} checkpoint does not exist: {fold_checkpoint}"
        for extra_task in range(5):
            fold_checkpoint = task3_primary_or_metastatic_dir / f"fold_{fold_num}_extra_task_{extra_task}.pth"
        print(f"✓ Fold {fold_num} checkpoint found")
    
    print("All data, labels and checkpoints are ready! ✓")

# Run the check
check_data_labels_checkpoints()


Checking data, labels and checkpoints...
Current directory: /data/dataserver01/zhangruipeng/code/BoneFM/release
Finetune directory: /data/dataserver01/zhangruipeng/code/BoneFM/release/finetune
✓ Fold 0 dataset: 125 subdirectories found
✓ Fold 1 dataset: 125 subdirectories found
✓ Fold 2 dataset: 125 subdirectories found
✓ Fold 3 dataset: 125 subdirectories found
✓ Fold 4 dataset: 125 subdirectories found
Checking image paths for fold 0 (6542 entries)...
✓ Fold 0 labels: All 6542 image files exist
Checking image paths for fold 1 (7498 entries)...
✓ Fold 1 labels: All 7498 image files exist
Checking image paths for fold 2 (6109 entries)...
✓ Fold 2 labels: All 6109 image files exist
Checking image paths for fold 3 (11163 entries)...
✓ Fold 3 labels: All 11163 image files exist
Checking image paths for fold 4 (6044 entries)...
✓ Fold 4 labels: All 6044 image files exist
✓ BoneFM backbone checkpoint found
✓ Fold 0 checkpoint found
✓ Fold 1 checkpoint found
✓ Fold 2 checkpoint found
✓ Fold 

## Step 1: Import required environment dependencies and set CUDA device
 
This section imports all necessary modules for inference and configures the CUDA device settings.

**Note**: Please ensure that the data, labels and checkpoint verification in Step 0 has passed before proceeding.


In [2]:
# Import basic environment dependencies for this open-source repository
import os
import argparse
from omegaconf import OmegaConf
import pathlib
# Set CUDA device number
# Requirements for CUDA device:
# - GPU memory > 24GB
# - CUDA version >= 12.4
# - Python version: 3.9
# Set CUDA device - ensure the device has sufficient GPU memory (>24GB recommended)
os.environ['CUDA_VISIBLE_DEVICES'] = '3'

# Import trainer module and initialize the specified trainer
import finetune.trainer as trainer



/data/dataserver01/zhangruipeng/code/BoneFM/release/finetune/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/data/dataserver01/zhangruipeng/code/BoneFM/release/finetune/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/data/dataserver01/zhangruipeng/code/BoneFM/release/finetune/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


## Step 2: Set dataset and checkpoint paths for each fold and experiment result save paths

This section configures the data paths, model checkpoint paths, and output directories for each fold in the 5-fold cross-validation setup. 


In [3]:
def run_primary_or_metastatic_inference(fold_num):
    """
    Run primary or metastatic inference for a specific fold
    
    Args:
        fold_num (int): The fold number for cross-validation (0-4)
    """
    print(f'Start evaluate on fold {fold_num}')
    
    # Create ArgumentParser object for command-line argument handling
    parser = argparse.ArgumentParser(description="DinoV2 Inference")

    # Add command line arguments for model configuration and output settings
    parser.add_argument("--config-file", type=str, default="finetune/configs/task3_primary_or_metastatic_eval.yaml", help="Model configuration file")
    parser.add_argument("--output-dir", default=f"finetune_logs/task3_primary_or_metastatic_inference_fold_{fold_num}", type=str, help="Output directory to write results and logs")
    parser.add_argument("--log-interval", type=int, help="Log interval", default=10)
    parser.add_argument("--log_display", action="store_true", help="Whether to display log", default=True)

    # Parse command line arguments, pass empty list to avoid reading from command line
    args = parser.parse_args([])

    # Load default configuration file
    default_config_path = pathlib.Path("finetune/configs/default_configs.yaml")
    default_cfg = OmegaConf.load(default_config_path)

    # Load user-specified configuration file
    user_cfg = OmegaConf.load(args.config_file)

    # Merge default and user configuration files with user config taking precedence
    merged_cfg = OmegaConf.merge(default_cfg, user_cfg)

    # Apply merged configuration to command line arguments
    # Only set attributes that don't exist or have None values
    for key, value in merged_cfg.items():
        # Convert key to string to ensure compatibility with hasattr/setattr
        key_str = str(key)
        if not hasattr(args, key_str):
            setattr(args, key_str, value)
        else:
            if getattr(args, key_str) is None:
                setattr(args, key_str, value)

    # Convert output directory to absolute path for consistency
    args.output_dir = os.path.abspath(args.output_dir)

    # Create output directory if it doesn't exist
    if not os.path.exists(args.output_dir):
        os.makedirs(args.output_dir)

    # Configure model checkpoint paths for the current fold
    args.model.backbone_ckpt_path = './finetune/checkpoints/BoneFM.pth'
    args.model.model_ckpt_dict['primary_metastatic'] = f'./finetune/checkpoints/task3_primary_or_metastatic/fold_{fold_num}.pth'
    args.model.model_ckpt_dict['osteolytic'] = f'./finetune/checkpoints/task3_primary_or_metastatic/fold_{fold_num}_extra_task_0.pth'
    args.model.model_ckpt_dict['osteoblastic'] = f'./finetune/checkpoints/task3_primary_or_metastatic/fold_{fold_num}_extra_task_1.pth'
    args.model.model_ckpt_dict['type_of_primary_tumor'] = f'./finetune/checkpoints/task3_primary_or_metastatic/fold_{fold_num}_extra_task_2.pth'
    args.model.model_ckpt_dict['pathological_fracture'] = f'./finetune/checkpoints/task3_primary_or_metastatic/fold_{fold_num}_extra_task_3.pth'
    args.model.model_ckpt_dict['spinal_cord_compression'] = f'./finetune/checkpoints/task3_primary_or_metastatic/fold_{fold_num}_extra_task_4.pth'
    
    # Get relative task list (exclude main task from relative tasks)
    args.model.relative_task = [task for task in args.model.relative_task if task != args.model.main_task]

    # Get main task checkpoint path
    args.model.main_task_ckpt_path = args.model.model_ckpt_dict[args.model.main_task]

    # Set dataset file paths for the current fold
    args.data.val_dataset = f'./finetune/data/labels/task3_primary_or_metastatic/fold_{fold_num}.csv'
    args.data.test_dataset = args.data.val_dataset

    # Inference rounds for 'primary or metastatic' task
    args.optim.epochs = 2

    # Save final configuration to YAML file for reproducibility
    args_dict = vars(args)
    sorted_args_dict = dict(sorted(args_dict.items()))

    config_yaml_path = os.path.join(args.output_dir, "config.yaml")
    with open(config_yaml_path, "w") as f:
        OmegaConf.save(config=OmegaConf.create(sorted_args_dict), f=f)

    print('Successfully loaded config file')

    
    trainer_names = sorted(name for name in trainer.__dict__ if 'Trainer' in name and callable(trainer.__dict__[name]))
     
    # Initialize trainer instance based on configuration
    primary_or_metastatic_eval_trainer = getattr(trainer, args.trainer)(args)
    
    # Execute the training/evaluation process
    primary_or_metastatic_eval_trainer.run()
    
    # Clean up GPU memory by deleting trainer instance
    del primary_or_metastatic_eval_trainer


# Step 3: Execute Inference for Each Fold Sequentially

This section performs primary or metastatic inference across all 5 folds of the cross-validation split. Each fold is processed sequentially to generate predictions on the respective test datasets.


In [4]:
# Iterate through 5-fold cross-validation for primary or metastatic inference
for fold_num in range(5):
    run_primary_or_metastatic_inference(fold_num)

Start evaluate on fold 0
Successfully loaded config file


2025-06-23 23:43 Test dataset: ./finetune/data/labels/task3_primary_or_metastatic/fold_0.csv / Length: 6542 / loader: 52


Use relative model for inference: osteolytic
Use relative model for inference: osteoblastic
Use relative model for inference: type_of_primary_tumor
Use relative model for inference: pathological_fracture
Use relative model for inference: spinal_cord_compression


2025-06-23 23:43 Use relative model for inference: dict_keys(['osteolytic', 'osteoblastic', 'type_of_primary_tumor', 'pathological_fracture', 'spinal_cord_compression'])
2025-06-23 23:45 Load backbone checkpoint from ./finetune/checkpoints/BoneFM.pth
2025-06-23 23:45 Load main task checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_0.pth
2025-06-23 23:45 Load relative model checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_0_extra_task_0.pth for osteolytic
2025-06-23 23:45 Load relative model checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_0_extra_task_1.pth for osteoblastic
2025-06-23 23:45 Load relative model checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_0_extra_task_2.pth for type_of_primary_tumor
2025-06-23 23:45 Load relative model checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_0_extra_task_3.pth for pathological_fracture
2025-06-23 23:45 Load relative model checkp

Start evaluate on fold 1
Successfully loaded config file


...

2025-06-24 03:27 Use relative model for inference: dict_keys(['osteolytic', 'osteoblastic', 'type_of_primary_tumor', 'pathological_fracture', 'spinal_cord_compression'])
2025-06-24 03:29 Load backbone checkpoint from ./finetune/checkpoints/BoneFM.pth
2025-06-24 03:29 Load main task checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_3.pth
2025-06-24 03:29 Load relative model checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_3_extra_task_0.pth for osteolytic
2025-06-24 03:29 Load relative model checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_3_extra_task_1.pth for osteoblastic
2025-06-24 03:29 Load relative model checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_3_extra_task_2.pth for type_of_primary_tumor
2025-06-24 03:29 Load relative model checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_3_extra_task_3.pth for pathological_fracture
2025-06-24 03:29 Load relative model checkp

Start evaluate on fold 4
Successfully loaded config file


2025-06-24 05:30 Test dataset: ./finetune/data/labels/task3_primary_or_metastatic/fold_4.csv / Length: 6044 / loader: 48


Use relative model for inference: osteolytic
Use relative model for inference: osteoblastic
Use relative model for inference: type_of_primary_tumor
Use relative model for inference: pathological_fracture
Use relative model for inference: spinal_cord_compression


2025-06-24 05:30 Use relative model for inference: dict_keys(['osteolytic', 'osteoblastic', 'type_of_primary_tumor', 'pathological_fracture', 'spinal_cord_compression'])
2025-06-24 05:33 Load backbone checkpoint from ./finetune/checkpoints/BoneFM.pth
2025-06-24 05:33 Load main task checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_4.pth
2025-06-24 05:33 Load relative model checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_4_extra_task_0.pth for osteolytic
2025-06-24 05:33 Load relative model checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_4_extra_task_1.pth for osteoblastic
2025-06-24 05:33 Load relative model checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_4_extra_task_2.pth for type_of_primary_tumor
2025-06-24 05:33 Load relative model checkpoint from ./finetune/checkpoints/task3_primary_or_metastatic/fold_4_extra_task_3.pth for pathological_fracture
2025-06-24 05:33 Load relative model checkp

# Results of Task 3: Primary or Metastatic Inference across 5 Folds

This section displays the inference results for primary or metastatic task evaluated across 5-fold cross-validation. The model performance is measured using AUROC (Area Under the Receiver Operating Characteristic curve) and AUPRC (Area Under the Precision-Recall Curve) metrics.


In [2]:
# Import metrics calculation function
from finetune.utils.utils import calculate_bonecot_metrics

# Set the directory path where prediction results are saved
pred_save_dir = "./finetune_logs/task3_primary_or_metastatic_inference_fold_0"

# Calculate AUROC and AUPRC metrics for primary or metastatic task
auroc, auprc = calculate_bonecot_metrics(pred_save_dir, task_name='primary_metastatic')

# Print the evaluation results
print(f"Task3 primary or metastatic AUROC: {auroc:.4f}, AUPRC: {auprc:.4f}")

Task3 primary or metastatic AUROC: 0.8554, AUPRC: 0.9155
